In [ ]:
"""
Memoryless: P(Xt+1|Xt, Xt-1n..., X1) = P(Xt+1|Xt),
(if some loans are always late (1+ month)
would the current state be the only indicator of future states?)
"""

"""
Time-homogeneous: Transition probabilities are constant over time,
does this hold in real life (portfolio of loans in a bearish, bullish economy)
"""

"Finite state space: System can only be in a finite number of states"

"""
Recurrence: A state is recurrent if the chain returns to it with probability 1

Irreducibility: A chain is irreducible if every state can be reached from every other state

Steady State: The long-term probabilities that stabilize as time approaches infinity

Absorbing States: States that once entered, cannot be left (probability of staying = 1)

Periodicity: The number of steps needed to return to a state (aperiodic if random)

Ergodicity: When a chain is both irreducible and aperiodic, leading to unique steady state

Transience: States that have a non-zero probability of never being revisited
"""

import numpy as np


P = np.array([
    [0.95, 0.05, 0.00, 0.00],
    [0.25, 0.60, 0.15, 0.00],
    [0.15, 0.10, 0.70, 0.05],
    [0.08, 0.00, 0.00, 0.92]
])

P_8 = np.linalg.matrix_power(P, 8)

states = ['Current', '30-60 Days', '60-90 Days', '90+ Days']

print("8-Month Transition Probabilities:")
print("---------------------------------")
for i, start_state in enumerate(states):
    print(f"\nStarting from {start_state}:")
    for j, end_state in enumerate(states):
        prob = P_12[i,j] * 100
        print(f"  → {end_state}: {prob:.2f}%")

In [ ]:
import numpy as np

P = np.array([
    [0.95, 0.05, 0.00, 0.00],
    [0.25, 0.60, 0.15, 0.00],
    [0.15, 0.10, 0.70, 0.05],
    [0.08, 0.00, 0.00, 0.92]
])

# play with the initial distribution of loans
initial_dist = np.array([.5, .2, 0, .3])

# play with number of transitions, will we converge to a certain probabilites of states
P_8 = np.linalg.matrix_power(P, 8)


final_dist = initial_dist @ P_8


print(f"Expected proportion of loans that will be 90+ days delinquent after 8 months: {final_dist[3]*100:.4f}%")

In [ ]:
# Calculate steady state distribution by iterating transition matrix
n_steps = 50
states = np.zeros((n_steps, 4))
states[0] = initial_dist

for t in range(1, n_steps):
    states[t] = states[t-1] @ P

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=list(range(n_steps)),
    y=states[:, 0],
    name='Current',
    line=dict(color='rgba(0, 255, 0, 0.8)', width=2)
))

fig.add_trace(go.Scatter(
    x=list(range(n_steps)),
    y=states[:, 1],
    name='30-59 Days Late',
    line=dict(color='rgba(255, 255, 0, 0.8)', width=2)
))

fig.add_trace(go.Scatter(
    x=list(range(n_steps)),
    y=states[:, 2],
    name='60-89 Days Late',
    line=dict(color='rgba(255, 165, 0, 0.8)', width=2)
))

fig.add_trace(go.Scatter(
    x=list(range(n_steps)),
    y=states[:, 3],
    name='90+ Days Late',
    line=dict(color='rgba(255, 0, 0, 0.8)', width=2)
))

fig.update_layout(
    height=600,
    width=1200,
    title='Convergence to Steady State Distribution',
    xaxis_title='Time Steps',
    yaxis_title='Proportion of Loans',
    plot_bgcolor='rgba(0,0,0,0)',
    paper_bgcolor='rgba(0,0,0,0)',
    font=dict(color='white'),
    showlegend=True,
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="center",
        x=0.5
    )
)

fig.update_xaxes(
    showgrid=True,
    gridwidth=1,
    gridcolor='rgba(128,128,128,0.2)',
    zeroline=True,
    zerolinewidth=1,
    zerolinecolor='rgba(128,128,128,0.5)'
)

fig.update_yaxes(
    showgrid=True,
    gridwidth=1,
    gridcolor='rgba(128,128,128,0.2)',
    zeroline=True,
    zerolinewidth=1,
    zerolinecolor='rgba(128,128,128,0.5)'
)

fig.show()

print("\nImportant Notes on Steady State Analysis:")
print("1. This steady state analysis assumes temporal homogeneity (transition probabilities")
print("   remain constant over time), which may not hold in reality due to:")
print("   - Economic cycles")
print("   - Seasonal effects")
print("   - Policy changes")
print("   - External shocks")
print("\n2. The convergence shown here assumes the Markov chain is:")
print("   - Irreducible (all states can be reached from all other states)")
print("   - Aperiodic (returns to states occur at irregular intervals)")
print("   Without these properties, a unique steady state may not exist.")

In [ ]:
"""
The natural next questions are:

What do we define as states for our Markov chain?
How do we find the probabilities for our transition matrix?
Defining states isn't too difficult and is largely problem dependent.

Some examples:

Volatility (Temperature): Low, Med, High
Market (Trend): Bullish, Bearish, Sideways
Liquidity (Spread): Low, Med, High
We can come up with various thresholds for each of these states using data from the market

VIX
SPX
Spreads
Ok, then how do we get probabilities?

______________________________________________________________________________________________________________________________________
Maximum Likelihood Estimation (MLE)
The method of maximum likelihood estimation is extremely intuitive

What is the data generating distribution that has the highest likelihood of producing the data we've seen historically?

There are several assumptions (both for Markov Chains & MLE) that enable this functionality, we will discuss them after observing the technique itself
"""

# Generate synthetic data from a normal distribution
np.random.seed(42)
n_samples = 1000
true_mean = 0
true_std = 1
data = np.random.normal(true_mean, true_std, n_samples)

# Calculate MLE parameters at different sample sizes
window_sizes = np.arange(10, n_samples, 10)
mle_means = np.zeros(len(window_sizes))
mle_stds = np.zeros(len(window_sizes))

for i, window in enumerate(window_sizes):
    sample = data[:window]
    mle_means[i] = np.mean(sample)
    mle_stds[i] = np.std(sample)

# Create figure for MLE convergence
fig = go.Figure()

# Plot histogram of full dataset
fig.add_trace(go.Histogram(
    x=data,
    name='Observed Data',
    histnorm='probability density',
    nbinsx=30,
    opacity=0.7,
    marker_color='rgba(128, 128, 128, 0.6)'
))

# Plot true distribution
x = np.linspace(-4, 4, 100)
true_pdf = 1/(true_std * np.sqrt(2*np.pi)) * np.exp(-(x-true_mean)**2/(2*true_std**2))
fig.add_trace(go.Scatter(
    x=x,
    y=true_pdf,
    name='True Distribution',
    line=dict(color='rgba(255, 0, 0, 0.8)', width=2, dash='dash')
))

# Plot final MLE fit
final_pdf = 1/(mle_stds[-1] * np.sqrt(2*np.pi)) * np.exp(-(x-mle_means[-1])**2/(2*mle_stds[-1]**2))
fig.add_trace(go.Scatter(
    x=x,
    y=final_pdf,
    name='MLE Fit',
    line=dict(color='rgba(0, 255, 0, 0.8)', width=2)
))

# Update layout
fig.update_layout(
    height=600,
    width=1200,
    title='Maximum Likelihood Estimation - Fitting Normal Distribution',
    xaxis_title='Value',
    yaxis_title='Probability Density',
    plot_bgcolor='rgba(0,0,0,0)',
    paper_bgcolor='rgba(0,0,0,0)',
    font=dict(color='white'),
    showlegend=True,
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="center",
        x=0.5
    )
)

# Update axes
fig.update_xaxes(
    showgrid=True,
    gridwidth=1,
    gridcolor='rgba(128,128,128,0.2)',
    zeroline=True,
    zerolinewidth=1,
    zerolinecolor='rgba(128,128,128,0.5)'
)

fig.update_yaxes(
    showgrid=True,
    gridwidth=1,
    gridcolor='rgba(128,128,128,0.2)',
    zeroline=True,
    zerolinewidth=1,
    zerolinecolor='rgba(128,128,128,0.5)'
)

fig.show()

print("\nKey Points about Maximum Likelihood Estimation for Normal Distribution:")
print("1. The histogram shows the actual distribution of observed data")
print("2. The red dashed line shows the true underlying distribution")
print("3. The green line shows our MLE fit, which closely matches the true distribution")
print("4. MLE finds the parameters (mean, std) that maximize the likelihood of observing our data")

"""
Key Points about Maximum Likelihood Estimation for Normal Distribution:
1. The histogram shows the actual distribution of observed data
2. The red dashed line shows the true underlying distribution
3. The green line shows our MLE fit, which closely matches the true distribution
4. MLE finds the parameters (mean, std) that maximize the likelihood of observing our data
"""

In [ ]:
# Generate synthetic data to demonstrate MLE
np.random.seed(42)
n_samples = 1000
true_probs = np.array([0.7, 0.2, 0.1])  # True probabilities for 3 states
states = np.random.choice(3, size=n_samples, p=true_probs)

# Calculate empirical probabilities over time
window_sizes = np.arange(10, n_samples, 10)
empirical_probs = np.zeros((len(window_sizes), 3))

for i, window in enumerate(window_sizes):
    counts = np.bincount(states[:window], minlength=3)
    empirical_probs[i] = counts / window

# Create figure for MLE convergence
fig = go.Figure()

# Plot convergence for each state probability
state_names = ['State A', 'State B', 'State C']
colors = ['rgba(0, 255, 0, 0.8)', 'rgba(255, 165, 0, 0.8)', 'rgba(255, 0, 0, 0.8)']

for i in range(3):
    # Plot empirical probabilities
    fig.add_trace(go.Scatter(
        x=window_sizes,
        y=empirical_probs[:, i],
        name=f'{state_names[i]} (Empirical)',
        line=dict(color=colors[i], width=2)
    ))
    
    # Plot true probabilities
    fig.add_trace(go.Scatter(
        x=[window_sizes[0], window_sizes[-1]],
        y=[true_probs[i], true_probs[i]],
        name=f'{state_names[i]} (True)',
        line=dict(color=colors[i], width=2, dash='dash')
    ))

# Update layout
fig.update_layout(
    height=600,
    width=1200,
    title='Maximum Likelihood Estimation - Convergence to True Probabilities',
    xaxis_title='Number of Samples',
    yaxis_title='Probability',
    plot_bgcolor='rgba(0,0,0,0)',
    paper_bgcolor='rgba(0,0,0,0)',
    font=dict(color='white'),
    showlegend=True,
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="center",
        x=0.5
    )
)

# Update axes
fig.update_xaxes(
    showgrid=True,
    gridwidth=1,
    gridcolor='rgba(128,128,128,0.2)',
    zeroline=True,
    zerolinewidth=1,
    zerolinecolor='rgba(128,128,128,0.5)'
)

fig.update_yaxes(
    showgrid=True,
    gridwidth=1,
    gridcolor='rgba(128,128,128,0.2)',
    zeroline=True,
    zerolinewidth=1,
    zerolinecolor='rgba(128,128,128,0.5)',
    range=[0, 1]
)

fig.show()

print("\nKey Points about Maximum Likelihood Estimation:")
print("1. As we collect more samples, our empirical probabilities (solid lines)")
print("   converge to the true probabilities (dashed lines)")
print("2. MLE provides consistent estimates - more data leads to better estimates")
print("3. The convergence rate follows the Law of Large Numbers")
print("4. Early estimates can be volatile due to small sample sizes")
"""
Key Points about Maximum Likelihood Estimation:
1. As we collect more samples, our empirical probabilities (solid lines)
converge to the true probabilities (dashed lines)
2. MLE provides consistent estimates - more data leads to better estimates
3. The convergence rate follows the Law of Large Numbers
4. Early estimates can be volatile due to small sample sizes
"""
